In [1]:
from agent import OrdersAgent
agent = await OrdersAgent.init_mcp()
graph = agent.build_graph()

/Users/anupam/Library/Mobile Documents/com~apple~CloudDocs/Documents/Documents - Anupam’s MacBook Air/Codebase/Studies/learning-aiml-principal/.venv/lib/python3.14/site-packages/langchain_core/utils/pydantic.py:41: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1 import BaseModel as BaseModelV1
/Users/anupam/Library/Mobile Documents/com~apple~CloudDocs/Documents/Documents - Anupam’s MacBook Air/Codebase/Studies/learning-aiml-principal/LangGraph/subagents/orders/agent.py:23: RuntimeWarning: coroutine 'OrdersAgent.init_mcp' was never awaited
  self.init_mcp()


In [2]:
from langgraph.types import Command
from uuid import uuid4

async def chat(query: str, thread_id: str | None = None):
    thread_id = thread_id or str(uuid4())
    config = {"configurable": {"thread_id": thread_id}}
    result = await graph.ainvoke(input={"query": query}, config=config)
    return thread_id, result

async def resume(thread_id: str, answer: str):
    config = {"configurable": {"thread_id": thread_id}}
    result = await graph.ainvoke(Command(resume=answer), config=config)
    return result

In [3]:
thread_id, result = await chat("What's up")

In [9]:
print(result["__interrupt__"][0].value['message'])

Which order is this about? 
[{'key': 'fd8f5312-2a31-4006-b21a-79cae597ae7b'}, {'key': 'f487a8f8-60d4-4107-be31-19ad733d5117'}, {'key': 'ec5f8170-8822-4144-94f7-e4df6be0d53d'}, {'key': '0c7b3c03-427d-4637-a195-aaaa49f049f0'}, {'key': '3488ed7b-eeec-4ce4-827c-4dc2cbeec5b1'}]


In [10]:
result = await resume(thread_id, "f487a8f8-60d4-4107-be31-19ad733d5117")   # answer the interrupt, same thread_id
result["__interrupt__"]   # check if there's a *next* interrupt (e.g. intent question)


[Interrupt(value={'type': 'input_request', 'message': 'Describe your issue in a few words'}, id='86e0046e540a3a0a6b836ca76be61c96')]

In [11]:
result = await resume(thread_id, "I want to track the order status")   # answer the interrupt, same thread_id
result

{'messages': [AIMessage(content='Here are your recent orders:\n1. Order fd8f5312-2a31-4006-b21a-79cae597ae7b - out_for_delivery - Chair,USB-C Cable,Monitor\n2. Order f487a8f8-60d4-4107-be31-19ad733d5117 - out_for_delivery - USB-C Cable\n3. Order ec5f8170-8822-4144-94f7-e4df6be0d53d - shipped - Chair,Monitor\n4. Order 0c7b3c03-427d-4637-a195-aaaa49f049f0 - shipped - Monitor,USB-C Cable\n5. Order 3488ed7b-eeec-4ce4-827c-4dc2cbeec5b1 - delivered - Monitor\nWhich order are you interested in?\n\nPlease enter the order ID', additional_kwargs={}, response_metadata={}, id='b53dc639-0a00-4717-8c3d-9eb62961df9c', tool_calls=[], invalid_tool_calls=[]),
  AIMessage(content="Your order status: {'status': 'out_for_delivery', 'delivery_date': '2026-07-23', 'delivery_address': 'Embassy Tech Village, Bengaluru, KA 560100'}", additional_kwargs={}, response_metadata={}, id='9b81b9fe-becd-4342-85f8-d74d45a8cc08', tool_calls=[], invalid_tool_calls=[])],
 'action': 'order-track',
 'order': {'key': UUID('f48

In [13]:
result['messages'][-1]

AIMessage(content="Your order status: {'status': 'out_for_delivery', 'delivery_date': '2026-07-23', 'delivery_address': 'Embassy Tech Village, Bengaluru, KA 560100'}", additional_kwargs={}, response_metadata={}, id='9b81b9fe-becd-4342-85f8-d74d45a8cc08', tool_calls=[], invalid_tool_calls=[])

In [22]:
# Here resume won't work because it does not have an interrupt

result = await chat(thread_id, "Can the delivery date be changed")   # answer the interrupt, same thread_id
result

('Can the delivery date be changed',
 {'messages': [AIMessage(content='Here are your recent orders:\n1. Order fd8f5312-2a31-4006-b21a-79cae597ae7b - out_for_delivery - Chair,USB-C Cable,Monitor\n2. Order f487a8f8-60d4-4107-be31-19ad733d5117 - out_for_delivery - USB-C Cable\n3. Order ec5f8170-8822-4144-94f7-e4df6be0d53d - shipped - Chair,Monitor\n4. Order 0c7b3c03-427d-4637-a195-aaaa49f049f0 - shipped - Monitor,USB-C Cable\n5. Order 3488ed7b-eeec-4ce4-827c-4dc2cbeec5b1 - delivered - Monitor\nWhich order are you interested in?\n\nPlease enter the order ID', additional_kwargs={}, response_metadata={}, id='a87455ef-8e86-4380-be37-2d536b506ee3', tool_calls=[], invalid_tool_calls=[]),
   AIMessage(content='Here are your recent orders:\n1. Order fd8f5312-2a31-4006-b21a-79cae597ae7b - out_for_delivery - Chair,USB-C Cable,Monitor\n2. Order f487a8f8-60d4-4107-be31-19ad733d5117 - out_for_delivery - USB-C Cable\n3. Order ec5f8170-8822-4144-94f7-e4df6be0d53d - shipped - Chair,Monitor\n4. Order 0c7b

In [23]:
result

('Can the delivery date be changed',
 {'messages': [AIMessage(content='Here are your recent orders:\n1. Order fd8f5312-2a31-4006-b21a-79cae597ae7b - out_for_delivery - Chair,USB-C Cable,Monitor\n2. Order f487a8f8-60d4-4107-be31-19ad733d5117 - out_for_delivery - USB-C Cable\n3. Order ec5f8170-8822-4144-94f7-e4df6be0d53d - shipped - Chair,Monitor\n4. Order 0c7b3c03-427d-4637-a195-aaaa49f049f0 - shipped - Monitor,USB-C Cable\n5. Order 3488ed7b-eeec-4ce4-827c-4dc2cbeec5b1 - delivered - Monitor\nWhich order are you interested in?\n\nPlease enter the order ID', additional_kwargs={}, response_metadata={}, id='a87455ef-8e86-4380-be37-2d536b506ee3', tool_calls=[], invalid_tool_calls=[]),
   AIMessage(content='Here are your recent orders:\n1. Order fd8f5312-2a31-4006-b21a-79cae597ae7b - out_for_delivery - Chair,USB-C Cable,Monitor\n2. Order f487a8f8-60d4-4107-be31-19ad733d5117 - out_for_delivery - USB-C Cable\n3. Order ec5f8170-8822-4144-94f7-e4df6be0d53d - shipped - Chair,Monitor\n4. Order 0c7b